[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CS7150/classdemos/blob/main/optimization/momentum.ipynb)

# Momentum: averaging out the noise

This notebook builds a small 1D loss landscape on purpose: a smooth, convex "bowl" with a high-frequency ripple riding on top of it. The ripple's amplitude is *not* constant &mdash; it grows in proportion to $|x|$, so the ripple is large far from the optimum and shrinks to nothing exactly at $x=0$. The loss looks like

$$L(x) = \tfrac{1}{2}a x^2 + c\,|x|\sin\!\left(\tfrac{2\pi x}{p}\right),$$

where $a$ controls how steep the bowl is, $c$ controls the ripple's amplitude scale, and $p$ is the ripple's period. The first term is the "real" signal we actually want to descend; the second term is high-frequency noise superimposed on it.

**Why plain gradient descent bounces.** Ordinary gradient descent takes the current gradient at face value on every step:

$$x \leftarrow x - \eta g, \qquad g = \nabla L(x).$$

Near a ripple tooth, $g$ is dominated by the ripple term, which flips sign every half period as $x$ crosses a peak or trough. So instead of making steady progress toward the optimum, plain GD's step direction flips back and forth from one iteration to the next, and it can settle into a stable back-and-forth orbit trapped inside a single ripple tooth &mdash; never actually reaching $x=0$ &mdash; even though the *underlying* bowl gradient is pushing it there.

**Why momentum's EMA cancels the ripple.** Momentum does not step in the direction of the raw gradient. Instead it keeps a running exponential moving average (EMA) of the gradient, updated at every step,

$$k \leftarrow \beta k + (1-\beta) g,$$

and then steps using that averaged quantity instead of the raw gradient:

$$x \leftarrow x - \eta k.$$

Because the ripple's contribution to $g$ alternates in sign roughly every half period while the bowl's contribution $ax$ keeps the same sign over many consecutive steps, averaging with a $\beta$ close to $1$ (so the average has a long effective memory, spanning many ripple periods) causes the alternating ripple terms to cancel each other out in the average, while the consistently-signed bowl term survives and accumulates. The net effect is that $k$ tracks the *slow* bowl gradient much more faithfully than any single noisy sample of $g$ does, so momentum rides smoothly down the bowl instead of getting stuck oscillating inside one tooth.

The two code cells below implement exactly these two update rules, with nothing hidden: given the same starting point, plain GD gets trapped near a ripple tooth while momentum reaches the optimum.

This matches the interactive version at [classdemos: momentum](https://cs7150.github.io/classdemos/demos/optimizers/momentum.html).

In [ ]:
#@title Setup: loss landscape + plotting helper (double-click to inspect) { display-mode: "form" }
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# Same landscape as the interactive demo's defaults.
ramp_amp = 0.05    # ripple amplitude scale
r_period = 0.22    # ripple period
bowl_a = 0.4       # bowl steepness

def loss(x):
    return 0.5 * bowl_a * x**2 + ramp_amp * np.abs(x) * np.sin(2 * np.pi * x / r_period)

def grad(x):
    th = 2 * np.pi * x / r_period
    return (bowl_a * x
            + ramp_amp * np.sign(x) * np.sin(th)
            + ramp_amp * np.abs(x) * (2 * np.pi / r_period) * np.cos(th))

def start_x(target=1.2):
    # Start just past the peak of a ripple tooth, on its downward slope,
    # so both trajectories begin at the edge of the same tooth.
    k = round(target / r_period - 0.25)
    peak = r_period * (k + 0.25)
    return peak - 0.08 * r_period

def plot_trajectories(paths, colors, labels):
    xs = np.linspace(-1.7, 1.7, 800)
    plt.figure(figsize=(7, 4))
    plt.plot(xs, 0.5 * bowl_a * xs**2, '--', color='0.6', label='bowl alone')
    plt.plot(xs, loss(xs), color='0.35', linewidth=1.6, label='bowl + ripple (real loss)')
    for path, color, label in zip(paths, colors, labels):
        path = np.array(path)
        plt.plot(path, loss(path), color=color, linewidth=2, marker='o', markersize=3, label=label)
    plt.xlabel('x')
    plt.ylabel('loss')
    plt.legend()
    plt.title('Plain GD vs. momentum on a rippled bowl')
    plt.show()

## Plain gradient descent

$$x \leftarrow x - \eta g$$

In [ ]:
eta = 0.1
n_steps = 300

x = start_x()
gd_path = [x]
for _ in range(n_steps):
    g = grad(x)
    x = x - eta * g
    gd_path.append(x)

## Momentum

$$k \leftarrow \beta k + (1-\beta) g$$
$$x \leftarrow x - \eta k$$

In [ ]:
beta = 0.99

x = start_x()
k = 0.0
mom_path = [x]
for _ in range(n_steps):
    g = grad(x)
    k = beta * k + (1 - beta) * g
    x = x - eta * k
    mom_path.append(x)

In [ ]:
plot_trajectories([gd_path, mom_path], ['tab:orange', 'tab:blue'], ['plain GD', 'momentum'])
print(f'final x: plain GD = {gd_path[-1]:.4f}, momentum = {mom_path[-1]:.4f}')